# Experiment A — Clique topology (το causal κλείσιμο του RQ2)

**Τι ελέγχουμε:** αν το hub deception (0% baseline → 33–100% adversarial) και το influence asymmetry είναι ιδιότητες της ΔΟΜΗΣ ή των ΜΟΝΤΕΛΩΝ.

**Σχεδιασμός:** 4-agent clique (όλοι βλέπουν όλους), PD+SH, 5 αποφασιστικά scenarios (no_comm+cheap_talk μέσω του baseline, counterfactual, framing_team, framing_competitive), N=5.

**Κανόνας απόφασης:** αν το exploitation πέσει >50% σχετικά στην clique → δομικό εύρημα. Αν επιμείνει → ιδιότητα μοντέλων.

**Ένα μοντέλο ανά session.** Τρέξε το notebook 3 φορές (Qwen2.5-7B, Gemma-2-9B, Llama-3.1-8B).

## Setup — install, GPU check, clone, HF token

1. Settings → Accelerator → **GPU T4 x2**
2. Settings → Internet → **On**
3. Add-ons → Secrets → `HF_TOKEN` (χρειάζεται για Llama/Gemma)

**Προσοχή:** το repo πρέπει να έχει γίνει push με τις αλλαγές του Phase 1.5 (topologies + `--action-retries`) πριν τρέξει αυτό το notebook.

In [ ]:
!pip install -q bitsandbytes python-dotenv

import torch
assert torch.cuda.is_available(), 'GPU not enabled!'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
GITHUB_REPO = 'https://github.com/stsimpe/cheaptalk_bench.git'
import os
if not os.path.exists('/kaggle/working/repo'):
    !git clone $GITHUB_REPO /kaggle/working/repo
%cd /kaggle/working/repo/cheaptalk_bench
# sanity: Phase-1.5 features present
assert 'clique' in open('topology.py').read(), 'Repo lacks Phase-1.5 topologies — push first!'
!ls

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HUGGINGFACE_API_KEY'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF token loaded')
except Exception as e:
    print('No HF_TOKEN secret (fine for Qwen):', e)

In [ ]:
MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct'
# MODEL_ID = 'google/gemma-2-9b-it'
# MODEL_ID = 'meta-llama/Llama-3.1-8B-Instruct'

In [ ]:
!python run_all_scenarios.py --provider local --model-id $MODEL_ID \
    --topology clique \
    --scenarios baseline counterfactual framing_team framing_competitive \
    --n-runs 5 --max-new-tokens 256 \
    --out-dir-base /kaggle/working/results \
    --zip-after-each --zip-mirror /kaggle/working

Τα zips εμφανίζονται στο `/kaggle/working` (Output panel). Κατέβασέ τα στο `diplomatikh/` ως νέο run folder, π.χ. `run6_<model>_clique/`.